In [4]:
import traci
import os
import sys
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

# ---------------- CLEAN START ----------------
if traci.isLoaded():
    traci.close()

# ---------------- SUMO SETUP ----------------
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    raise Exception("SUMO_HOME not set")

sumoBinary = "sumo-gui"
sumoCmd = [sumoBinary, "-c", "../sumo_files/simple.sumocfg"]

traci.start(sumoCmd)

tls = traci.trafficlight.getIDList()[0]
print("Using TLS:", tls)

# ---------------- STATE (LANE-WISE) ----------------
def get_state(tls_id):
    lanes = traci.trafficlight.getControlledLanes(tls_id)
    state = []
    for lane in lanes:
        state.append(traci.lane.getLastStepVehicleNumber(lane))
    return state  # list

state_size = len(get_state(tls))
print("State size:", state_size)

# ---------------- REWARD ----------------
def get_reward(state):
    return -sum(state)

# ---------------- DQN MODEL ----------------
class DQN(nn.Module):
    def __init__(self, input_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model = DQN(state_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# ---------------- MEMORY ----------------
memory = deque(maxlen=2000)

# ---------------- PARAMETERS ----------------
gamma = 0.9
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995

# ---------------- ACTION ----------------
def choose_action(state):
    global epsilon

    if random.random() < epsilon:
        return random.choice([0, 1])

    state_tensor = torch.FloatTensor([state])  # vector
    q_values = model(state_tensor)

    return torch.argmax(q_values).item()

# ---------------- REPLAY ----------------
def replay():
    if len(memory) < 32:
        return

    batch = random.sample(memory, 32)

    for state, action, reward, next_state in batch:
        state_tensor = torch.FloatTensor([state])
        next_state_tensor = torch.FloatTensor([next_state])

        q_values = model(state_tensor)
        next_q_values = model(next_state_tensor)

        target = q_values.clone().detach()
        target[0][action] = reward + gamma * torch.max(next_q_values).item()

        loss = loss_fn(q_values, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# ---------------- SIMULATION ----------------
current_phase = 0

print("Running Lane-wise DQN...")

for step in range(2000):
    traci.simulationStep()

    state = get_state(tls)
    action = choose_action(state)

    # Apply action
    if action == 1:
        current_phase = 2 if current_phase == 0 else 0
        traci.trafficlight.setPhase(tls, current_phase)

    reward = get_reward(state)
    next_state = get_state(tls)

    # Store experience
    memory.append((state, action, reward, next_state))

    # Train
    replay()

    # Decay epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    print(f"Step: {step} | State: {state} | Action: {action} | Reward: {reward} | Epsilon: {epsilon:.3f}")

traci.close()

Using TLS: A1
State size: 9
Running Lane-wise DQN...
Step: 0 | State: [0, 0, 0, 0, 0, 0, 0, 0, 0] | Action: 1 | Reward: 0 | Epsilon: 0.995
Step: 1 | State: [0, 0, 0, 0, 0, 0, 0, 0, 0] | Action: 1 | Reward: 0 | Epsilon: 0.990
Step: 2 | State: [0, 0, 0, 0, 0, 0, 0, 0, 0] | Action: 1 | Reward: 0 | Epsilon: 0.985
Step: 3 | State: [0, 0, 0, 0, 0, 0, 0, 0, 0] | Action: 0 | Reward: 0 | Epsilon: 0.980
Step: 4 | State: [0, 0, 0, 1, 1, 1, 0, 0, 0] | Action: 1 | Reward: -3 | Epsilon: 0.975
Step: 5 | State: [1, 1, 1, 1, 1, 1, 0, 0, 0] | Action: 1 | Reward: -6 | Epsilon: 0.970
Step: 6 | State: [1, 1, 1, 1, 1, 1, 1, 1, 1] | Action: 0 | Reward: -9 | Epsilon: 0.966
Step: 7 | State: [1, 1, 1, 1, 1, 1, 1, 1, 1] | Action: 1 | Reward: -9 | Epsilon: 0.961
Step: 8 | State: [2, 2, 2, 1, 1, 1, 1, 1, 1] | Action: 0 | Reward: -12 | Epsilon: 0.956
Step: 9 | State: [2, 2, 2, 1, 1, 1, 1, 1, 1] | Action: 0 | Reward: -12 | Epsilon: 0.951
Step: 10 | State: [2, 2, 2, 1, 1, 1, 1, 1, 1] | Action: 0 | Reward: -12 | Epsil